# MAP, MLE, and Regularization: One Identity to Bind Them

Three threads from this lesson — maximum likelihood (notebook 04: likelihood of a fit = product of Gaussians on the distances), regularization (notebook 05: loss + L2 penalty, with a cliffhanger), and MAP (notebook 06: multiply the likelihood by a prior) — braid into a single identity. Give every model a probability by drawing its coefficients from a standard normal; then maximizing $P(\text{data} \mid \text{model})\,P(\text{model})$ *is* minimizing $\text{square loss} + \text{L2 penalty}$. The bridge between the two worlds is a logarithm, which turns products of Gaussians into sums of squares. It is based on the closing video of Lesson 6 (Point Estimation), which ties the whole lesson together.

## 1. Introduction

Notebook 05 ended on a promise: the L2 penalty $\sum a_i^2$ *is* Bayesian probability in disguise. This notebook pays it off. The missing ingredient is a precise meaning for "the probability of a model" — and once coefficients are drawn from a standard normal, everything clicks: complex models with big coefficients are *a priori* improbable, and their log-improbability is exactly the regularization term.

After completing this notebook you will be able to:

- Re-rank candidate models by $P(\text{data} \mid \text{model}) \times P(\text{model})$ and explain why the winner changes.
- Define the likelihood of a model as the product of standard-normal densities of its coefficients.
- Run the log derivation: likelihood × prior $\;\to\;$ square loss + L2 penalty, keeping track of what happens to constants and signs.
- Translate freely between the two vocabularies: likelihood ↔ loss, prior ↔ regularizer, maximize ↔ minimize.
- Extend the dictionary: prior variance ↔ regularization strength $\lambda$, Laplace prior ↔ L1 penalty.

## 2. Theory

### Three models, one more contest

Three models could have generated the data (think of the line, the quadratic, and the wild degree-10 polynomial from notebook 05). Each generates the observed points with some probability — a Gaussian standing at every point of the curve, generating data close to it, as in notebook 04:

$$
P(\text{data} \mid \text{model}_1) \;<\; P(\text{data} \mid \text{model}_2) \;<\; P(\text{data} \mid \text{model}_3)
$$

By likelihood alone the most complex model wins — it hugs every point. But just like the popcorn-throwing contest of notebook 06, we must also ask *how likely each model was in the first place*. Simple models are likely; slightly complex ones less so; wild ones very unlikely:

$$
P(\text{model}_1) \;>\; P(\text{model}_2) \;>\; P(\text{model}_3)
$$

Multiply the two, and the winner is no longer the third model — perhaps it is the second:

$$
\text{pick} \;\; \arg\max_{\text{model}} \; P(\text{data} \mid \text{model}) \, P(\text{model})
$$

### The elephant in the room: what is $P(\text{model})$?

A model is its coefficients. Take

$$
\text{model}_1: \; y = a_1 x, \qquad
\text{model}_2: \; y = a_1 x + a_2 x^2, \qquad
\text{model}_{10}: \; y = a_1 x + \cdots + a_{10} x^{10}
$$

and declare that each coefficient is picked from a **standard normal distribution**. Then the likelihood of one coefficient $a_i$ is its density,

$$
\frac{1}{\sqrt{2\pi}} \, e^{-\frac{1}{2} a_i^2}
$$

and the likelihood of a whole model is the **product over its coefficients**:

$$
P(\text{model}) \;=\; \prod_{i} \frac{1}{\sqrt{2\pi}} \, e^{-\frac{1}{2} a_i^2}
$$

Two forces make complex models improbable: every extra coefficient contributes another factor $\le \frac{1}{\sqrt{2\pi}} \approx 0.4$, and every *large* coefficient contributes a crushing $e^{-a_i^2/2}$.

### The other factor, from notebook 04

If the model's curve misses the data points by vertical distances $d_1, \dots, d_n$ (Gaussian noise with unit variance), then

$$
P(\text{data} \mid \text{model}) \;=\; \prod_{j=1}^{n} \frac{1}{\sqrt{2\pi}} \, e^{-\frac{1}{2} d_j^2}
$$

### Logarithms: products become sums

Maximize the product of the two factors. Products are unwieldy; logarithms turn them into sums (and preserve the argmax, since $\log$ is increasing):

$$
\log\big[P(\text{data} \mid \text{model})\,P(\text{model})\big]
= \underbrace{-\tfrac{1}{2}\textstyle\sum_j d_j^2}_{\text{from the data term}}
\;\underbrace{-\tfrac{1}{2}\textstyle\sum_i a_i^2}_{\text{from the model term}}
\;+\; \text{constants}
$$

The constants (all those $\log\frac{1}{\sqrt{2\pi}}$) don't depend on the coefficients — drop them. Multiply by $-2$, which flips maximization into **minimization**:

$$
\boxed{\;\arg\max \; P(\text{data} \mid \text{model})\,P(\text{model})
\;=\;
\arg\min \; \underbrace{\textstyle\sum_j d_j^2}_{\text{square loss}} + \underbrace{\textstyle\sum_i a_i^2}_{\text{L2 penalty}}\;}
$$

That is where the regularization term comes from. Training a regularized regression **is** Bayesian MAP estimation.

### The dictionary

| Bayesian world (maximize) | Machine-learning world (minimize) |
|---|---|
| likelihood $P(\text{data} \mid \text{model})$ | square loss $\sum_j d_j^2$ |
| prior $P(\text{model})$ | L2 penalty $\sum_i a_i^2$ |
| product of probabilities | sum of loss terms (via $\log$) |
| maximize the posterior (MAP) | minimize loss + penalty |
| flat prior (MLE) | no regularization ($\lambda = 0$) |

One refinement the exercises develop: with noise variance $\sigma_d^2$ and coefficient prior variance $\sigma_a^2$, the identity becomes $\sum d_j^2 + \lambda \sum a_i^2$ with $\lambda = \sigma_d^2 / \sigma_a^2$ — a *tighter* prior (small $\sigma_a$) means *stronger* regularization, and a Laplace prior instead of a Gaussian yields the L1 penalty.

## 3. Worked Examples

### Example 1: the full derivation for a two-coefficient model

Model $y = a_1 x + a_2 x^2$, five data points missed by distances $d_1, \dots, d_5$:

$$
P(\text{data} \mid \text{model})\,P(\text{model}) =
\underbrace{\prod_{j=1}^{5} \tfrac{1}{\sqrt{2\pi}} e^{-\frac{1}{2}d_j^2}}_{\text{distances}}
\;\times\;
\underbrace{\tfrac{1}{\sqrt{2\pi}} e^{-\frac{1}{2}a_1^2} \cdot \tfrac{1}{\sqrt{2\pi}} e^{-\frac{1}{2}a_2^2}}_{\text{coefficients}}
$$

Take the log and drop the seven constant terms $\log\frac{1}{\sqrt{2\pi}}$:

$$
-\tfrac{1}{2}\big(d_1^2 + \cdots + d_5^2\big) - \tfrac{1}{2}\big(a_1^2 + a_2^2\big)
$$

Multiplying by $-2$ and switching to minimization gives the regularized loss $\;\sum_j d_j^2 + (a_1^2 + a_2^2)$.

### Example 2: how improbable is a complex model?

Concrete coefficients, standard-normal prior:

- $y = 2x$: $\;P = \frac{1}{\sqrt{2\pi}}e^{-2} \approx 0.054$
- $y = 2x + 0.5x^2$: $\;P = \left(\frac{1}{\sqrt{2\pi}}\right)^2 e^{-\frac{1}{2}(4 + 0.25)} \approx 0.019$
- a degree-10 model with all $|a_i| = 2$: $\;P = \left(\frac{1}{\sqrt{2\pi}}\right)^{10} e^{-\frac{1}{2} \cdot 40} \approx 2 \times 10^{-13}$

Each extra or bigger coefficient multiplies in another small factor — the prior's version of "punish complexity."

### Example 3: where $\lambda$ comes from

Redo the derivation with noise $d_j \sim N(0, \sigma_d^2)$ and coefficients $a_i \sim N(0, \sigma_a^2)$. The log becomes $-\frac{1}{2\sigma_d^2}\sum_j d_j^2 - \frac{1}{2\sigma_a^2}\sum_i a_i^2 + \text{const}$; multiplying by $-2\sigma_d^2$ gives

$$
\sum_j d_j^2 + \underbrace{\frac{\sigma_d^2}{\sigma_a^2}}_{\lambda} \sum_i a_i^2
$$

The regularization parameter from notebook 05 is a **ratio of variances**: believing coefficients are small (small $\sigma_a$) cranks $\lambda$ up; noisy data (big $\sigma_d$) also cranks it up, because the data deserves less trust. Standard normals on both sides give the video's $\lambda = 1$.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(3)

GAUSS_CONST = 1 / np.sqrt(2 * np.pi)

def coefficient_prior(coeffs):
    """P(model): product of standard-normal densities of the coefficients."""
    coeffs = np.asarray(coeffs, dtype=float)
    return np.prod(GAUSS_CONST * np.exp(-0.5 * coeffs**2))

### The log identity, symbolically

SymPy expands $\log\big[\prod_j \frac{1}{\sqrt{2\pi}}e^{-d_j^2/2} \cdot \prod_i \frac{1}{\sqrt{2\pi}}e^{-a_i^2/2}\big]$ and shows that, constants aside, it is $-\frac{1}{2}(\text{square loss} + \text{L2 penalty})$.

In [ ]:
d = sp.symbols("d1:6", real=True)          # five distances
a = sp.symbols("a1:3", real=True)          # two coefficients

gauss = lambda v: sp.exp(-v**2 / 2) / sp.sqrt(2 * sp.pi)
product = sp.prod([gauss(dj) for dj in d]) * sp.prod([gauss(ai) for ai in a])

log_product = sp.expand_log(sp.log(product), force=True)
square_loss = sum(dj**2 for dj in d)
l2_penalty = sum(ai**2 for ai in a)

difference = sp.simplify(log_product - (-(square_loss + l2_penalty) / 2))
print("log(likelihood x prior) + (loss + penalty)/2 =", difference)
print("-> a pure constant: the log identity holds, constants aside.")

### Example 2's numbers, computed

In [ ]:
models = [("y = 2x", [2]),
          ("y = 2x + 0.5x^2", [2, 0.5]),
          ("degree 10, all |a_i| = 2", [2] * 10)]

for name, coeffs in models:
    print(f"P(model) for {name:28s} = {coefficient_prior(coeffs):.3g}")

### Regularized regression = MAP, verified

Five points from a gentle quadratic. Objective A: minimize $\sum d_j^2 + a_1^2 + a_2^2$ — plain ridge regression, solved in closed form by $(X^\top X + \lambda I)^{-1} X^\top y$ with $\lambda = 1$. Objective B: maximize $P(\text{data}\mid\text{model})\,P(\text{model})$ over a grid of $(a_1, a_2)$. Same winner, to grid precision.

In [ ]:
x_pts = np.array([0.5, 1.0, 1.5, 2.0, 2.5])
y_pts = 1.2 * x_pts + 0.4 * x_pts**2 + rng.normal(0, 0.5, size=x_pts.size)

X = np.column_stack([x_pts, x_pts**2])     # model y = a1 x + a2 x^2

# Objective A: ridge closed form, lambda = 1
ridge_coeffs = np.linalg.solve(X.T @ X + np.eye(2), X.T @ y_pts)

# Objective B: brute-force MAP over a grid of (a1, a2)
a1_grid, a2_grid = np.meshgrid(np.linspace(-1, 3, 401), np.linspace(-1, 3, 401))
predictions = x_pts[:, None, None] * a1_grid + (x_pts**2)[:, None, None] * a2_grid
distances_sq = ((y_pts[:, None, None] - predictions) ** 2).sum(axis=0)
log_likelihood = -0.5 * distances_sq        # + constants, which never matter
log_prior = -0.5 * (a1_grid**2 + a2_grid**2)
log_posterior = log_likelihood + log_prior

best = np.unravel_index(np.argmax(log_posterior), log_posterior.shape)
map_coeffs = np.array([a1_grid[best], a2_grid[best]])

print(f"ridge (loss + penalty) minimizer: a1 = {ridge_coeffs[0]:.3f}, a2 = {ridge_coeffs[1]:.3f}")
print(f"MAP (likelihood x prior) maximizer: a1 = {map_coeffs[0]:.3f}, a2 = {map_coeffs[1]:.3f}")
print(f"same point (to the grid step of 0.01): "
      f"{np.allclose(ridge_coeffs, map_coeffs, atol=0.01)}")

### The model contest, re-ranked by the prior

Fit polynomials of degree 1, 2, and 10 to a dozen points from a quadratic. By likelihood alone (equivalently: by raw square loss) the degree-10 model wins. Add the log-prior — the L2 penalty in disguise — and the quadratic takes the contest, exactly the video's flip.

In [ ]:
x_data = np.linspace(0, 3, 12)
y_data = 1.0 * x_data + 0.5 * x_data**2 + rng.normal(0, 0.6, size=x_data.size)

contest = {}
for degree in (1, 2, 10):
    coeffs = np.polyfit(x_data, y_data, degree)
    residuals = y_data - np.polyval(coeffs, x_data)
    log_lik = np.sum(np.log(GAUSS_CONST) - 0.5 * residuals**2)
    log_prior = np.sum(np.log(GAUSS_CONST) - 0.5 * coeffs**2)
    contest[degree] = (coeffs, log_lik, log_prior)

print(f"{'model':>10s} {'log-likelihood':>15s} {'log-prior':>12s} {'sum (log joint)':>16s}")
for degree, (coeffs, log_lik, log_prior) in contest.items():
    print(f"degree {degree:>3d} {log_lik:>15.1f} {log_prior:>12.1f} {log_lik + log_prior:>16.1f}")

by_lik = max(contest, key=lambda k: contest[k][1])
by_joint = max(contest, key=lambda k: contest[k][1] + contest[k][2])
print(f"\nlikelihood alone picks degree {by_lik}; likelihood x prior picks degree {by_joint}.")

## 5. Visualizations

### The prior pulls the peak toward the origin

Left: the likelihood surface over $(a_1, a_2)$ — MLE lives at its peak. Right: the posterior surface — the same hill multiplied by a prior centered at $(0,0)$, so the peak slides toward the origin. That slide *is* regularization shrinking the coefficients.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), sharey=True)

mle_best = np.unravel_index(np.argmax(log_likelihood), log_likelihood.shape)
mle_coeffs = np.array([a1_grid[mle_best], a2_grid[mle_best]])

for ax, surface, peak, title in [
        (axes[0], log_likelihood, mle_coeffs, "log-likelihood (MLE view)"),
        (axes[1], log_posterior, map_coeffs, "log-posterior = log-lik + log-prior (MAP view)")]:
    levels = np.linspace(surface.max() - 12, surface.max(), 25)   # zoom on the hilltop
    ax.contourf(a1_grid, a2_grid, surface, levels=levels, cmap="viridis",
                extend="min")
    ax.contour(a1_grid, a2_grid, surface, levels=levels[::3], colors="white",
               linewidths=0.5, alpha=0.5)
    ax.plot(*peak, "r*", markersize=15,
            label=f"peak ({peak[0]:.2f}, {peak[1]:.2f})")
    ax.plot(0, 0, "wo", markersize=6, label="prior center (0, 0)")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(r"$a_1$")
    ax.legend(loc="lower left", fontsize=9)
axes[0].set_ylabel(r"$a_2$")
plt.suptitle("Multiplying by the prior slides the peak toward small coefficients")
plt.tight_layout()
plt.show()

### The contest as bars: likelihood up, prior down, joint decides

The log scale makes the multiplication additive — literally stacking the two bars.

In [ ]:
degrees = list(contest)
log_liks = [contest[k][1] for k in degrees]
log_priors = [contest[k][2] for k in degrees]
log_joints = [l + p for l, p in zip(log_liks, log_priors)]

x = np.arange(len(degrees))
plt.figure(figsize=(9.5, 5))
for offset, values, color, label in [
        (-0.25, log_liks, "tab:blue", "log-likelihood (fit quality)"),
        (0.0, log_priors, "tab:red", "log-prior (simplicity)"),
        (0.25, log_joints, "tab:green", "sum = log joint (the decider)")]:
    plt.bar(x + offset, values, 0.25, color=color, label=label)
    for xi, v in zip(x + offset, values):
        va = "top" if v > -100 else "bottom"   # huge bars: label inside the bar
        plt.annotate(f"{v:,.1f}", xy=(xi, v), ha="center", va=va,
                     fontsize=8, rotation=90)
plt.axhline(0, color="black", linewidth=0.8)
plt.yscale("symlog")                       # the degree-10 prior is ~ -3e8
plt.xticks(x, [f"degree {k}" for k in degrees])
plt.ylabel("log probability (higher = better; symlog scale)")
plt.title("Degree 10 wins the likelihood but its prior is catastrophic; the quadratic wins the joint")
plt.grid(True, alpha=0.3, axis="y")
plt.legend(fontsize=9, loc="lower left")
plt.show()

### Prior width is the $\lambda$ dial

Ridge fits of the two-coefficient model under priors of different widths: a tight prior ($\sigma_a$ small, $\lambda$ big) flattens the fit toward zero; a wide prior ($\lambda \to 0$) recovers plain least squares — MLE.

In [ ]:
x_line = np.linspace(0, 2.7, 200)
X_line = np.column_stack([x_line, x_line**2])

plt.figure(figsize=(9.5, 5))
plt.scatter(x_pts, y_pts, color="black", zorder=5, label="data")

for sigma_a, color in [(0.1, "tab:red"), (1.0, "tab:green"), (100.0, "tab:blue")]:
    lam = 0.5**2 / sigma_a**2                # lambda = sigma_d^2 / sigma_a^2
    coeffs = np.linalg.solve(X.T @ X + lam * np.eye(2), X.T @ y_pts)
    plt.plot(x_line, X_line @ coeffs, color=color, linewidth=2,
             label=f"prior width {sigma_a:g} ->  $\\lambda$ = {lam:.2g}  "
                   f"(a1 = {coeffs[0]:.2f}, a2 = {coeffs[1]:.2f})")

plt.xlabel("x")
plt.ylabel("y")
plt.title(r"Tight prior = strong regularization: $\lambda = \sigma_d^2 / \sigma_a^2$")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=9)
plt.show()

### The whole story on one page

Everything above in a single sheet: the model contest scored by likelihood and prior, the prior pulling the peak toward zero, the dictionary between MAP and regularization, and prior width as the λ dial.

<img src="img/11_map_regularization.png" alt="Concept sheet for this notebook" width="900">

## 6. Exercises

### Basic

**Exercise 1.** Under the standard-normal prior, compute $P(\text{model})$ for $y = 3x$ and for $y = 0.5x$. Which is more probable a priori, and by what factor?

**Exercise 2.** Fill in the dictionary from memory: in the identity $\arg\max P(\text{data}\mid\text{model})P(\text{model}) = \arg\min \sum d_j^2 + \sum a_i^2$, which Bayesian factor becomes the square loss, which becomes the penalty, what mathematical operation turns the products into sums, and why does maximizing become minimizing?

### Intermediate

**Exercise 3.** Rerun the derivation with $d_j \sim N(0, \sigma_d^2)$ and $a_i \sim N(0, \sigma_a^2)$ (by hand or with SymPy) and show the objective becomes $\sum d_j^2 + \lambda \sum a_i^2$ with $\lambda = \sigma_d^2/\sigma_a^2$. Sanity-check the two limits: $\sigma_a \to \infty$ and $\sigma_a \to 0$.

**Exercise 4.** *L1 from a different prior.* Suppose each coefficient instead follows a Laplace distribution, density $\frac{1}{2b}e^{-|a_i|/b}$. Show that the log derivation now yields the **L1 penalty** $\sum_i |a_i|$ (lasso). Plot the two negative log-priors, $\frac{1}{2}a^2$ vs $|a|$, and explain from the picture why the Laplace prior is happier to set coefficients *exactly* to zero.

### Advanced

**Exercise 5.** Prove the ridge–MAP equivalence in general: for the model $y = Xw$ with unit-variance Gaussian noise and prior $w \sim N(0, \sigma_a^2 I)$, show the MAP estimate is $\hat w = (X^\top X + \lambda I)^{-1} X^\top y$ with $\lambda = 1/\sigma_a^2$, by setting the gradient of the log-posterior to zero. Verify numerically on the five-point dataset for a $\lambda$ of your choice.

**Exercise 6.** *Data swamps the prior (again).* For datasets of growing size $n$ drawn from the same quadratic, track which degree wins the joint contest (degrees 1, 2, 10) and how far the ridge coefficients sit from the least-squares ones. Confirm the lesson-wide moral: as $n$ grows, MAP → MLE and the regularization term's influence fades.

## 7. Solutions

In [ ]:
# Exercise 1: prior probability of two one-coefficient models
p_3 = coefficient_prior([3])
p_05 = coefficient_prior([0.5])
print(f"Exercise 1: P(y = 3x)   = {p_3:.4f}")
print(f"            P(y = 0.5x) = {p_05:.4f}")
print(f"            ratio = {p_05 / p_3:.1f}x in favor of the small coefficient")
print(f"            (exact factor: e^(0.5*(9 - 0.25)) = {np.exp(0.5 * (9 - 0.25)):.1f})")

In [ ]:
# Exercise 2: the dictionary
print("Exercise 2:")
print("  likelihood P(data | model) --log--> -1/2 * square loss  (sum d_j^2)")
print("  prior      P(model)        --log--> -1/2 * L2 penalty   (sum a_i^2)")
print("  logarithm turns the products of Gaussians into sums;")
print("  multiplying the log by -2 flips the sign, so the argMAX of the")
print("  probabilities is the argMIN of loss + penalty.")

In [ ]:
# Exercise 3: general variances with sympy
sigma_d, sigma_a = sp.symbols("sigma_d sigma_a", positive=True)
dj, ai = sp.symbols("d_j a_i", real=True)

gauss_var = lambda v, s: sp.exp(-v**2 / (2 * s**2)) / (s * sp.sqrt(2 * sp.pi))
log_terms = sp.expand_log(sp.log(gauss_var(dj, sigma_d) * gauss_var(ai, sigma_a)),
                          force=True)
objective = sp.simplify(-2 * sigma_d**2 * log_terms)
print("Exercise 3: -2 sigma_d^2 * log(term_j * term_i) =")
sp.pprint(sp.collect(sp.expand(objective), (dj**2, ai**2)))
print("\nPer-term: d_j^2 + (sigma_d^2/sigma_a^2) a_i^2 + constants -> lambda = sigma_d^2/sigma_a^2.")
print("Limits: sigma_a -> infinity gives lambda -> 0 (flat prior, plain MLE);")
print("        sigma_a -> 0 gives lambda -> infinity (coefficients forced to 0).")

In [ ]:
# Exercise 4: Laplace prior -> L1 penalty
# log[ (1/2b) e^(-|a|/b) ] = -|a|/b - log(2b): dropping constants and scaling,
# the penalty is sum |a_i| -- the L1 (lasso) penalty.
a_axis = np.linspace(-3, 3, 400)

plt.figure(figsize=(8, 4.2))
plt.plot(a_axis, 0.5 * a_axis**2, color="tab:blue", linewidth=2,
         label=r"Gaussian: $\frac{1}{2}a^2$ (L2 / ridge)")
plt.plot(a_axis, np.abs(a_axis), color="tab:orange", linewidth=2,
         label=r"Laplace: $|a|$ (L1 / lasso)")
plt.xlabel("coefficient a")
plt.ylabel("negative log-prior (penalty)")
plt.title("Exercise 4: near zero, L2 goes flat but L1 keeps a sharp kink")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

print("Near a = 0 the quadratic penalty is almost flat -- shrinking further buys")
print("almost nothing -- while |a| keeps a constant slope all the way to zero,")
print("so the L1 optimum happily lands at exactly a = 0: sparse models.")

In [ ]:
# Exercise 5: gradient proof + numerical check
# log-posterior (constants dropped): -1/2 |y - Xw|^2 - 1/(2 sigma_a^2) |w|^2
# gradient: X^T (y - Xw) - w / sigma_a^2 = 0
#   =>  (X^T X + lambda I) w = X^T y,   lambda = 1 / sigma_a^2
lam = 2.0
w_ridge = np.linalg.solve(X.T @ X + lam * np.eye(2), X.T @ y_pts)

sigma_a_val = 1 / np.sqrt(lam)
log_post = (-0.5 * ((y_pts[:, None, None]
                     - x_pts[:, None, None] * a1_grid
                     - (x_pts**2)[:, None, None] * a2_grid) ** 2).sum(axis=0)
            - (a1_grid**2 + a2_grid**2) / (2 * sigma_a_val**2))
best5 = np.unravel_index(np.argmax(log_post), log_post.shape)

print(f"Exercise 5 (lambda = {lam}):")
print(f"  closed form  : a1 = {w_ridge[0]:.3f}, a2 = {w_ridge[1]:.3f}")
print(f"  grid argmax  : a1 = {a1_grid[best5]:.3f}, a2 = {a2_grid[best5]:.3f}")
print("  The stationarity condition (X^T X + lambda I) w = X^T y is exactly the")
print("  ridge normal equation -- MAP and ridge are the same estimator.")

In [ ]:
# Exercise 6: growing n -- MAP converges to MLE, prior influence fades
import warnings

# degree-10 fits are ill-conditioned by design; silence polyfit's complaint
# (np.RankWarning moved to np.exceptions.RankWarning in numpy 2.0)
RankWarning = getattr(np, "RankWarning", None) or np.exceptions.RankWarning
warnings.filterwarnings("ignore", category=RankWarning)

rng_ex6 = np.random.default_rng(11)
print(f"{'n':>5s} {'joint winner':>13s} {'|ridge - OLS| (2-coef model)':>30s}")
for n in (12, 30, 100, 300, 1000):
    x_n = np.linspace(0.1, 3, n)
    y_n = 1.0 * x_n + 0.5 * x_n**2 + rng_ex6.normal(0, 0.6, size=n)

    scores = {}
    for degree in (1, 2, 10):
        coeffs = np.polyfit(x_n, y_n, degree)
        residuals = y_n - np.polyval(coeffs, x_n)
        scores[degree] = (np.sum(np.log(GAUSS_CONST) - 0.5 * residuals**2)
                          + np.sum(np.log(GAUSS_CONST) - 0.5 * coeffs**2))
    winner = max(scores, key=scores.get)

    X_n = np.column_stack([x_n, x_n**2])
    w_ols = np.linalg.solve(X_n.T @ X_n, X_n.T @ y_n)
    w_map = np.linalg.solve(X_n.T @ X_n + np.eye(2), X_n.T @ y_n)
    gap = np.linalg.norm(w_map - w_ols)
    print(f"{n:>5d} {f'degree {winner}':>13s} {gap:>30.4f}")

print("\nThe joint contest settles on the true degree 2, and the ridge-vs-OLS gap")
print("shrinks toward 0: with enough data, the likelihood swamps the prior and")
print("MAP converges to MLE -- the same moral as the coin flips, now for models.")

## 8. Conclusion

- Model selection à la Bayes: maximize $P(\text{data} \mid \text{model}) \times P(\text{model})$ — fit quality times plausibility. Complex curves win the first factor and lose the second.
- **$P(\text{model})$ made precise**: draw each coefficient from a standard normal; the model's likelihood is the product $\prod_i \frac{1}{\sqrt{2\pi}}e^{-a_i^2/2}$, so extra and larger coefficients are exponentially penalized.
- **The log identity**: $\log$ turns the product of Gaussians into $-\frac{1}{2}\sum d_j^2 - \frac{1}{2}\sum a_i^2$ plus constants; dropping constants and multiplying by $-2$, *maximizing the posterior is minimizing square loss + L2 penalty*. Regularized regression **is** MAP estimation.
- The dictionary generalizes: prior variance sets $\lambda = \sigma_d^2/\sigma_a^2$ (tight prior ⇒ strong regularization), a Laplace prior yields the L1/lasso penalty, a flat prior yields plain MLE, and growing data makes MAP converge to MLE.

And with that, the lesson's three storylines — MLE, MAP, and regularization — are one story. This closes the lectures of the lesson; next up in the course: this week's exploratory data analysis lab and quiz.